<img src="imgs/AWS_logo_RGB.png" style="max-width:10%" align="right">
<img src="imgs/Grupo_Nutresa_logo.png" style="max-width:25%" align="left">

# Creación del feature store para Nielsen

Este script se encarga de:

- Leer los features necesarios de las tablas del datalake
- Hacer transformaciones generales y transversales a los features

<a id='contents' />

## Tabla de contenidos

1. [Carga de librerías](#loading)
2. [Pruebas Funcionales Scripts](#scripts)
3. [Ejecutar Script Completo](#py)
5. [Ejecutar Script desde Processor](#processor)
6. [Creación de SageMaker pipeline de features](#pipeline)

<a id='loading' />

## Carga de librerías
[(back to top)](#contents)

In [3]:
%load_ext autoreload
%autoreload 2
%run config_notebook.py

Directorio "src" adicionado para cargar modulos
Librerías boto3 y sagemaker importadas
Variables de sagemaker listas para ser usadas:
	* sagemaker_session
	* sagemaker_role: arn:aws:iam::544644514035:role/ause1-rol-ml-sagemaker-crud-processes
	* sagemaker_bucket: machine-learning-serviciosnutresa-modelos-lab
	* feature_store_bucket: machine-learning-serviciosnutresa-featurestore-lab


In [4]:
import json
import pandas as pd
import sagemaker
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.feature_store.feature_group import (FeatureGroup, FeatureTypeEnum, FeatureDefinition, DataCatalogConfig)
from datetime import datetime
import ast
import awswrangler as wr
from utilities import Utils
import time
from sagemaker.workflow.parameters import (ParameterInteger,ParameterString)
from sagemaker.processing import FeatureStoreOutput
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.workflow.steps import ProcessingStep
from sagemaker.workflow.pipeline import Pipeline

region = 'us-east-1'
s3 = boto3.resource('s3')
sm_feature_store = sagemaker_session.boto_session.client("sagemaker-featurestore-runtime", region_name=region)
ssm = boto3.client('ssm')

Definición de variables

In [5]:
# Variables para la creación del feature group
entidad = 'producto'
fuente = 'nielsen'
periodicidad = 'monthly'
complejidad = 'simple'
record_identifier_name = 'tag'
record_event_time_name = 'EventTime'

# Definición de nombre del feature group
feature_group_name = f'fg-{entidad}-{fuente}-{periodicidad}-{complejidad}'
feature_group_description = f'Features de la fuente {fuente} de {entidad}'

# Definición de tablas qeu alimentarán al feature group
db_datalake='db_nutresa_datalake_stage_dev'
tb_productos_preparados='mundos_tb_nielsen_productos_preparados_delta'
tb_entrenamiento='mundos_tb_nielsen_productos_entrenamiento_delta'

In [7]:
# Crear archivo de requirements.txt en el repositorio con determinadas librerías
%%writefile ../src/{fuente}/requirements.txt
sagemaker == 2.49.0
awswrangler == 2.4.0

Overwriting ../src/nielsen/requirements.txt


In [8]:
# Carga de archivo desde el repo a S3
s3.meta.client.upload_file(f'../src/{fuente}/requirements.txt', 
                           feature_store_bucket, 
                           f'artefactos/codigo/{fuente}/requirements.txt')

In [ ]:
s3.meta.client.upload_file(f'../src/{fuente}/processing_features.py', 
                           feature_store_bucket, 
                           f'artefactos/codigo/{fuente}/main.py')

In [6]:
# Donde esta almacenado los scripts de código y archivo de requirements.
requirements_path=f's3://{feature_store_bucket}/artefactos/codigo/{fuente}/requirements.txt'
code_path=f's3://{feature_store_bucket}/artefactos/codigo/{fuente}/main.py'

In [9]:
# Cada archivo fuente para el procesamiento debe estar ubicado dentro del contenedor en una carpeta independiente. Aquí se crean las carpetas para simular el comportamiento dentro del docker.
!sudo mkdir /opt/ml/processing
!sudo mkdir /opt/ml/processing/input
!sudo mkdir /opt/ml/processing/input/requirements

!sudo aws s3 cp s3://{feature_store_bucket}/artefactos/codigo/{fuente}/requirements.txt /opt/ml/processing/input/requirements/

mkdir: cannot create directory ‘/opt/ml/processing’: File exists
mkdir: cannot create directory ‘/opt/ml/processing/input’: File exists
mkdir: cannot create directory ‘/opt/ml/processing/input/requirements’: File exists
download: s3://machine-learning-serviciosnutresa-featurestore-lab/artefactos/codigo/nielsen/requirements.txt to ../../../../../opt/ml/processing/input/requirements/requirements.txt


<a id='scripts' />

## Pruebas Funcionales Scripts
[(back to top)](#contents)

Se prueba cada una de las funciones creadas en el script de forma independiente

In [10]:
import nielsen.processing_features as p

#### Recuperar parámetro SSM de columnas

In [ ]:
columns_definition = ast.literal_eval(ssm.get_parameter(Name='/ml/misc/features_daneshapes/columns_definition')['Parameter']['Value'])

In [ ]:
campos=columns_definition[fuente]

### Cargar dataset

In [ ]:
dataset = p.load_dataset(fuente, db_datalake, tb_productos_preparados, tb_entrenamiento, sagemaker_bucket, record_identifier_name, campos, record_event_time_name)

In [ ]:
dataset.head()

In [ ]:
dataset.mundo_experto.value_counts()

### Build feature definitions

In [ ]:
definitions=p.build_feature_definitions(dataset)

In [ ]:
definitions

### Obtener el feature group

In [73]:
print('Obteniendo el feature group... ')
sagemaker_feature_group = FeatureGroup(name=feature_group_name, 
                                       #feature_definitions=definitions, 
                                       sagemaker_session = sagemaker_session)

Obteniendo el feature group... 


### Crear el feature store
A partir de la definición del feature group

In [ ]:
sagemaker_role

In [ ]:
p.create_feature_store(sagemaker_feature_group=sagemaker_feature_group,
                      dataset=dataset,
                      role_arn=sagemaker_role,
                      sagemaker_bucket=sagemaker_bucket,
                      record_identifier_name=record_identifier_name,
                      record_event_time_name=record_event_time_name,
                      entidad=entidad,
                      feature_group_name=feature_group_name,
                      feature_group_description=feature_group_description)

### Checkear que el feature_group si haya sido creado

In [ ]:
# Checkear que el feature_group si haya sido creado
feature_group_status = sagemaker_feature_group.describe().get("FeatureGroupStatus")

In [ ]:
feature_group_status

In [ ]:
feature_group_status_offiline = sagemaker_feature_group.describe().get("OfflineStoreStatus")
feature_group_status_offiline

### Cargar datos al feature store. 
Cada worker corresponde a configuración diferente de la instancia, ej: 1 worker -> 4 vCPUs of compute capacity and 16 GB of memory

In [ ]:
sagemaker_feature_group.ingest(data_frame=dataset, max_workers=1, wait=True)

### Consultar Datos en el Feature Store Online

In [ ]:
# Consultar el feature group (solamente si el online store está habilitado)
tag = 'P000000000002007694000000002000036694760'
record_identifier=tag
sample_record = sm_feature_store.get_record(FeatureGroupName=feature_group_name, RecordIdentifierValueAsString=str(record_identifier))
sample_record

In [ ]:
#Date from the record
datetime.fromtimestamp(round(float(sample_record['Record'][-1]['ValueAsString']),0))

### Describir la metadata del feature store

In [ ]:
sagemaker_feature_group.describe()

### Borrar un feature group

Por la naturaleza inmutable de los feature groups, cuando hay un cambio en el esquema del feature group es necesario primero borrar el feature group existente

Esta definición de esquema es la que debería estar en la función de __build_feature_definitions__

In [74]:
sagemaker_feature_group.delete()

El feature group puede tomar unos segundos en ser borrado. Una vez no se pueda hacer una descripción de este, quiere decir que fue borrado.

In [ ]:
try:
    sagemaker_feature_group.describe()
    print('feature group still exists')
except:
    print('feature group deleted')

<a id='py' />

## Ejecutar Python desde el Script Completo
[(back to top)](#contents)

In [ ]:
# Cada uno de los parámetro definidos en el script son pasados, si son variables se debe anteponer el caracter $
!python ../src/{fuente}/processing_features.py --role-arn $sagemaker_role --feature-store-bucket $feature_store_bucket \
--entidad $entidad \
--fuente $fuente --periodicidad $periodicidad --record-identifier-name $record_identifier_name \
--record-event-time-name $record_event_time_name \
--complejidad $complejidad \
--db-datalake $db_datalake \
--tb-productos-preparados $tb_productos_preparados \
--tb-entrenamiento $tb_entrenamiento 

<a id='processor' />

## Ejecutar Script desde Processor
[(back to top)](#contents)

Ejecutar usando un SKLearnProcessor.

In [75]:
feature_store_bucket

'machine-learning-serviciosnutresa-featurestore-lab'

In [38]:
# Acá se instancia el processor
 # instance_type='local' se utiliza en forma de prueba, para utilizar la instancia para que proceso, no deja registro en sagemaker Processing jobs. Es mucho más rápido puesto que no levanta el docker
 # Si se requiere dejar visibilidad en Sagemaker Processing Jobs cambiar por 'ml.m5.xlarge'
sklearn_processor = SKLearnProcessor(framework_version='0.23-1',
                                     role=sagemaker_role,
                                     instance_type='local',
                                     instance_count=1,
                                     base_job_name=f'feature-{fuente}-{entidad}',
                                     volume_size_in_gb=5)

In [40]:
# Ejecuta el processor con los parámetros necesarios.
sklearn_processor.run(code=code_path,
                      inputs=[ProcessingInput(
                              source=requirements_path,
                              destination='/opt/ml/processing/input/requirements')],
                      arguments=['--region', 'us-east-1',
                                '--role-arn', sagemaker_role, 
                                '--feature-store-bucket', feature_store_bucket,
                                '--entidad', entidad,
                                '--fuente', fuente,
                                '--periodicidad', periodicidad,
                                '--record-identifier-name', record_identifier_name,
                                '--record-event-time-name', record_event_time_name,
                                '--complejidad', complejidad,
                                '--db-datalake', db_datalake,
                                '--tb-productos-preparados', tb_productos_preparados,
                                '--tb-entrenamiento', tb_entrenamiento],                   
                      wait=False) # wait=True. La celda se bloquea hasta que no termine el processor. False libera las celdas.

In [ ]:
# Revisar estado de Processing job:
sklearn_processor.latest_job.describe()['ProcessingJobStatus']

In [ ]:
#Saber cual nombre tiene el feature store
sklearn_processor.latest_job.job_name

<a id='pipeline' />

## SageMaker Pipeline
[(back to top)](#contents)

SageMaker Pipelines soporta la creación de DAGs de pasos y condiciones para orquestar trabajos de procesamiento de SageMaker y creación de recursos. En esta sección definimos un conjunto de parámetros de los pipelines, y definimos el paso de procesamiento en donde se crea el Feature Store y se realiza la ingesta de datos al mismo.

### Definir parámetros para la ejecución del pipeline

La definición de parámetros permite realizar ejecuciones diferentes del pipeline, sin tener que modificar la definición del pipeline como tal.

Los tipos de parámetros soportados son:

- ParameterString - representa el tipo str de Python
- ParameterInteger - representa el tipo int de Python
- ParameterFloat - representa el tipo float de Python

Estos parámetros permiten agregar un valor por defecto, que puede ser sobre-escrito en la ejecución del pipeline.
El valor por defecto especificado debe ser una instancia del tipo de parámetro. 

In [64]:
processing_instance_count = ParameterInteger(name="ProcessingInstanceCount",default_value=1)
processing_instance_type = ParameterString(name="ProcessingInstanceType",default_value="ml.m5.xlarge")
processing_db_datalake = ParameterString(name="ProcessingDBDatalake")
processing_tb_productos_preparados = ParameterString(name="ProcessingTBProductosPreparados")
processing_tb_entrenamiento = ParameterString(name="ProcessingTBEntrenamiento")

Instanciar el processor con los parámetros definidos

In [65]:
framework_version = '0.23-1'
sklearn_processor_features = SKLearnProcessor(framework_version=framework_version,
                                     role=sagemaker_role,
                                     instance_type=processing_instance_type,
                                     instance_count=processing_instance_count,
                                     base_job_name=f'feature-{fuente}',
                                     volume_size_in_gb=5)

Definir los pasos del pipeline. En este caso puentual es un sólo paso de tipo **ProcessingStep**

In [66]:
step_process = ProcessingStep(
    name=f'feature-{fuente}', # Nombre del processing step 
    processor=sklearn_processor_features,
    inputs=[
        ProcessingInput(source=requirements_path, 
                        destination='/opt/ml/processing/input/requirements')
    ],
    job_arguments=['--region', region,
                    '--role-arn', sagemaker_role, 
                    '--feature-store-bucket', feature_store_bucket ,
                    '--entidad', entidad,
                    '--fuente', fuente,
                    '--periodicidad', periodicidad,
                    '--record-identifier-name', record_identifier_name,
                    '--record-event-time-name', record_event_time_name,
                    '--complejidad', complejidad,
                    '--db-datalake', processing_db_datalake,
                    '--tb-productos-preparados', processing_tb_productos_preparados,
                    '--tb-entrenamiento', processing_tb_entrenamiento],
    code=code_path
)

### Definición del SageMaker Pipeline

En esta sección se crea el Pipeline a ejecutar. Se requieren los parámetros: `name`, `parameters`y `steps`. 

- El nombre `name` debe ser único en la región y la cuenta. 
- Todos los parámetros usados en las definiciones deben estar presentes.
- Los pasos en el pipeline no necesariamente deben estar listados en el mismo orden de ejecución.
- Los pasos deben ser únicos dentro de la lista de pasos del Pipeline.

**Definición del pipeline:**

In [67]:
pipeline_name = f'ml-features-mundos-{fuente}'
pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        processing_instance_type,
        processing_instance_count,
        processing_db_datalake,
        processing_tb_productos_preparados,
        processing_tb_entrenamiento
    ],
    steps=[step_process],
)

En el siguiente paso se genera el json del pipeline definido anteriormente, esta definición es usada para la creación del pipeline cuando se cree la arquitectura como código

In [68]:
definition = json.loads(pipeline.definition())
definition

{'Version': '2020-12-01',
 'Metadata': {},
 'Parameters': [{'Name': 'ProcessingInstanceType',
   'Type': 'String',
   'DefaultValue': 'ml.m5.xlarge'},
  {'Name': 'ProcessingInstanceCount', 'Type': 'Integer', 'DefaultValue': 1},
  {'Name': 'ProcessingDBDatalake', 'Type': 'String'},
  {'Name': 'ProcessingTBProductosPreparados', 'Type': 'String'},
  {'Name': 'ProcessingTBEntrenamiento', 'Type': 'String'}],
 'PipelineExperimentConfig': {'ExperimentName': {'Get': 'Execution.PipelineName'},
  'TrialName': {'Get': 'Execution.PipelineExecutionId'}},
 'Steps': [{'Name': 'feature-nielsen',
   'Type': 'Processing',
   'Arguments': {'ProcessingResources': {'ClusterConfig': {'InstanceType': {'Get': 'Parameters.ProcessingInstanceType'},
      'InstanceCount': {'Get': 'Parameters.ProcessingInstanceCount'},
      'VolumeSizeInGB': 5}},
    'AppSpecification': {'ImageUri': '683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:0.23-1-cpu-py3',
     'ContainerArguments': ['--region',
     

**Subir el Pipeline a SageMaker**

El rol pasado por parámetro será utilizado para crear todos los trabajos de SageMaker definidos en los pasos.

In [69]:
# upsert actualiza la definición del pipeline
pipeline.upsert(role_arn=sagemaker_role)

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:544644514035:pipeline/ml-features-mundos-nielsen',
 'ResponseMetadata': {'RequestId': '7c99aae8-f944-4258-a253-0979316f4c91',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '7c99aae8-f944-4258-a253-0979316f4c91',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '94',
   'date': 'Thu, 12 Aug 2021 21:31:09 GMT'},
  'RetryAttempts': 0}}

Ejecutar el pipeline con los parámetros pasados por defecto

### Ejecución del pipeline

**Ejecución con parámetros específicos**

Es posible correr ejecuciones especificando parámetros diferentes a los parámetros por defecto. Para esto se usa el argumento `parameters`, que es un diccionario que contiene los nombres de los parámetros y valores, y puede sobre-escribir los valores por defecto. 

In [70]:
execution = pipeline.start(
    parameters=dict(
        ProcessingDBDatalake=db_datalake,
        ProcessingTBProductosPreparados=tb_productos_preparados,
        ProcessingTBEntrenamiento=tb_entrenamiento
    )
)

In [72]:
#Estado del pipeline
execution.describe()['PipelineExecutionStatus']

'Succeeded'

La forma de eliminar un pipeline es por SDK con boto3

In [51]:
sm_cliente = boto3.client('sagemaker')
response = sm_cliente.delete_pipeline(
    PipelineName='feature-interna'
) 

Revisar los pasos definidos en el pipeline

In [52]:
execution.list_steps()

[{'StepName': 'feature-nielsen',
  'StartTime': datetime.datetime(2021, 8, 12, 19, 18, 16, 988000, tzinfo=tzlocal()),
  'StepStatus': 'Executing',
  'Metadata': {'ProcessingJob': {'Arn': 'arn:aws:sagemaker:us-east-1:544644514035:processing-job/pipelines-70e5zyvdnjjs-feature-nielsen-6euhsthysx'}}}]

**Artefactos generados por el pipeline:**

In [ ]:
viz = LineageTableVisualizer(sagemaker.session.Session())
for execution_step in reversed(execution.list_steps()):
    print(execution_step)
    display(viz.show(pipeline_execution_step=execution_step))
    time.sleep(5)

### Revisión de la creación de la tabla del feature group. 
La tabla del feature group esta conformada por el nombre del feature group más una serie de consecutivos, estos consecutivos cambian cada vez que se borra y vuelve a crear un feature group, por lo cual es importante no quemar este nombre en el código sino hallarlo programaticamente.

In [ ]:
nielsen_fg = FeatureGroup(name=feature_group_name, sagemaker_session=sagemaker_session)  
nielsen_query = nielsen_fg.athena_query()
nielsen_table = nielsen_query.table_name
query_string = f'SELECT * FROM "{nielsen_table}" limit 10'
db_feature_store="sagemaker_featurestore"

In [ ]:
# Before extracting the data we need to check if the feature store was populated
offline_store_contents = None
while offline_store_contents is None:    
    nielsen_total_record_count = Utils.get_historical_record_count(feature_group_name)
    if nielsen_total_record_count >= 1:
        print('[Features are available in Offline Store!]')
        offline_store_contents = nielsen_total_record_count
    else:
        print('[Waiting for data in Offline Store ...]')
        time.sleep(60)

In [ ]:
df = wr.athena.read_sql_query(query_string, database=db_feature_store, ctas_approach=False, s3_output = 's3://sagemaker-datalab-serviciosnutresa/athena/')
df.head()